## Tool definition

In [ ]:
%run langchain_common.py

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [ ]:
tool1.invoke({"x": 467})

In [ ]:
square_root.invoke({"x": 467})

## Adding to agents

In [ ]:
agent = create_agent(llm_noreason, tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [ ]:
question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

pp(response['messages'][-1].content)

In [ ]:
pp(response['messages'])

In [ ]:
#Retrieve the tool calls from the LLM response
response["messages"][1].tool_calls

## create_agent (under the hood)

In [ ]:
# Equivalent to create_agent, but using the bind_tools method on the LLM

llm_with_tools = llm.bind_tools(tools=[tool1])
query = "What is the square root of 900?"
response = llm_with_tools.invoke(query)
pp(response.tool_calls)

## agent.invoke (under the hood)

In [ ]:
query = "What is the square root of 1000?"
ai_msg = llm_with_tools.invoke(query)

pp(ai_msg)

for tool_call in ai_msg.tool_calls:
    # Look up the actual function from our 'tools' list
    # (In a real app, you'd use a dictionary mapping names to functions)
    selected_tool = {"square_root": square_root}[tool_call["name"].lower()]
    
    # 3. Execute the function with the arguments provided by the LLM
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # 4. Create a ToolMessage to send the result back to the LLM
    # This must include the tool_call_id so the LLM knows which request this satisfies
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=tool_call["id"]
    )

# 5. Feed the result back to the LLM so it can give a final natural language answer
final_response = llm_with_tools.invoke([query, ai_msg, tool_message])
pp(final_response.content)